In [2]:
import glob
import pandas as pd
import numpy as np

## This code scales data from intan voltage (0 - 3.3V) to the applied voltage (±5V) and converts it to degree s

In [8]:
# Load the background Python functions that allow for data loading and plotting
%run "./load-rhd-notebook-python-main/importrhdutilities.py"

Exception: File `'\'"./load-rhd-notebook-python-main/importrhdutilities.py"\''` not found.

In [3]:
#scaling- ±5V motor signal was converted to 0-3.3V by hardware circuit before recording (adjust accordinly for different setup)
ADC_MIDPOINT = 1.65        # 0V motor = 1.65V ADC (fixed hardware midpoint)
ADC_RANGE = 3.3            # ADC full range (0 to 3.3V- the one we saw)
MOTOR_RANGE = 10.0         # ±5V = 10V full range
scale_factor = MOTOR_RANGE / ADC_RANGE  # ~3.03

scaling_info = pd.DataFrame({
    'parameter': ['ADC_midpoint', 'ADC_range', 'motor_range', 'scale_factor'],
    'value':     [ADC_MIDPOINT,   ADC_RANGE,   MOTOR_RANGE,   scale_factor],
    'units':     ['V',            'V',          'V',           'V/V']})
scaling_info.to_csv('./dataFolder/3_Scaled_Stimulus/scaling_info.csv', index=False)

rhd_files = glob.glob('./dataFolder/1_RHD_Files/*.rhd')
print(f'Found {len(rhd_files)} files')

for filepath in rhd_files:
    print(f'Processing: {filepath}')
    result, data_present = load_file(filepath)
    
    name = filepath.split('/')[-1].replace('.rhd', '')

    # Subtract fixed hardware midpoint (1.65V) then scale back to ±5V
    stimulus_scaled = (result['board_adc_data'][0] - ADC_MIDPOINT) * scale_factor

    # Save individually per moth
    moth_df = pd.DataFrame({'scaled_voltage': stimulus_scaled})
    moth_df.to_csv('/Users/meghnaanand/Desktop/Analysis/' + name + '_scaledvoltage.csv', index=False)
    print(f'  Saved: {name}_scaled_voltage.csv')
    print(f'  Voltage range: {stimulus_scaled.min():.4f} to {stimulus_scaled.max():.4f} V')

print('All done!')

In [ ]:
df = pd.read_csv('/Users/meghnaanand/Desktop/Analysis/m23_wn2_scaledvoltage.csv')
print(df['scaled_voltage'].min())
print(df['scaled_voltage'].max())
print(df.head())

In [ ]:
files = glob.glob('/Users/meghnaanand/Desktop/Analysis/*_scaled_voltage.csv')
for f in files:
    df = pd.read_csv(f)
    name = f.split('/')[-1].replace('_scaled_voltage.csv', '')
    print(f'{name}: {df["scaled_voltage"].min():.4f} to {df["scaled_voltage"].max():.4f} V')